# 📦 BƯỚC 1: Thu thập & Hiểu dữ liệu
**Amazon Clothing Review Analysis & Recommendation**

---
### 🎯 Mục tiêu của Bước 1:
1. Hiểu cấu trúc file dữ liệu `.jsonl.gz`
2. Khám phá các cột (fields) có trong dataset
3. Xem dữ liệu thực tế trông như thế nào
4. Nắm được thống kê cơ bản (số dòng, missing values...)

> ⚠️ **Lưu ý**: File gốc rất lớn (~22.6M reviews). Bước này chỉ đọc **5,000 dòng đầu** để khám phá nhanh, KHÔNG load toàn bộ.

## Cell 1.1 — Import thư viện

In [ ]:
# ============================================================
# CELL 1.1: Import thư viện
# - pandas, numpy: xử lý dữ liệu dạng bảng
# - json, gzip   : đọc file .jsonl.gz
# - os, sys      : thao tác file/thư mục
# - matplotlib   : vẽ biểu đồ
# ============================================================

import pandas as pd
import numpy as np
import json
import gzip
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Thêm thư mục gốc vào path để import config
ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

print("✅ Libraries imported successfully!")
print(f"📁 Working directory: {Path().resolve()}")

## Cell 1.2 — Khai báo đường dẫn

In [ ]:
# ============================================================
# CELL 1.2: Khai báo đường dẫn tới các file
# Dùng Path() để tương thích cả Windows lẫn Mac/Linux
# ============================================================

# Thư mục gốc của dự án (amazon_clothing_project/)
ROOT_DIR      = Path().resolve().parent

# Đường dẫn file RAW
REVIEW_PATH   = ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz"
META_PATH     = ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz"

# Thư mục output
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra file có tồn tại không
print("🔍 Kiểm tra file dữ liệu:")
for name, path in [("Review", REVIEW_PATH), ("Meta", META_PATH)]:
    size_gb = path.stat().st_size / (1024**3) if path.exists() else 0
    status  = f"✅ Tìm thấy ({size_gb:.2f} GB)" if path.exists() else "❌ KHÔNG tìm thấy!"
    print(f"  {name}: {status}")
    print(f"         Path: {path}")

## Cell 1.3 — Đọc thử vài dòng để hiểu cấu trúc JSON

In [ ]:
# ============================================================
# CELL 1.3: Đọc thử RAW JSON để hiểu cấu trúc
# File .jsonl.gz = mỗi dòng là 1 JSON object riêng biệt
# Ta chỉ đọc 3 dòng đầu để xem structure
# ============================================================

print("=" * 60)
print("📄 CẤU TRÚC FILE REVIEW (3 dòng đầu - dạng JSON thô)")
print("=" * 60)

with gzip.open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line)
        print(f"\n--- Dòng {i+1} ---")
        for key, value in data.items():
            # Rút gọn text dài để dễ đọc
            display_val = str(value)[:80] + "..." if len(str(value)) > 80 else value
            print(f"  [{key}]: {display_val}")

print("\n" + "=" * 60)
print("📄 CẤU TRÚC FILE META (3 dòng đầu - dạng JSON thô)")
print("=" * 60)

with gzip.open(META_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line)
        print(f"\n--- Dòng {i+1} ---")
        for key, value in data.items():
            display_val = str(value)[:80] + "..." if len(str(value)) > 80 else value
            print(f"  [{key}]: {display_val}")

## Cell 1.4 — Load mẫu 5000 dòng để khám phá

In [ ]:
# ============================================================
# CELL 1.4: Load 5,000 dòng đầu để khám phá
# Tại sao chỉ 5000? Vì mục tiêu bước này là HIỂU dữ liệu,
# không cần load hết → nhanh hơn, không tốn RAM
# ============================================================

def peek_jsonl_gz(filepath, n_rows=5000):
    """Load n_rows đầu tiên từ file .jsonl.gz thành DataFrame"""
    data = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n_rows:
                break
            try:
                data.append(json.loads(line.strip()))
            except json.JSONDecodeError:
                continue  # Bỏ qua dòng lỗi
    return pd.DataFrame(data)

print("📦 Đang load 5,000 dòng đầu...")
df_review_peek = peek_jsonl_gz(REVIEW_PATH, n_rows=5000)
df_meta_peek   = peek_jsonl_gz(META_PATH,   n_rows=5000)

print(f"✅ Review sample shape : {df_review_peek.shape}")
print(f"✅ Meta sample shape   : {df_meta_peek.shape}")

## Cell 1.5 — Phân tích cột của Review dataset

In [ ]:
# ============================================================
# CELL 1.5: Phân tích chi tiết từng cột của Review
# Ta cần biết:
#   - Có những cột gì?
#   - Kiểu dữ liệu của mỗi cột?
#   - Có bao nhiêu % missing?
#   - Giá trị mẫu trông như thế nào?
# ============================================================

print("=" * 65)
print("📊 REVIEW DATASET - PHÂN TÍCH TỪNG CỘT")
print("=" * 65)

print("\n1️⃣ Danh sách cột:")
print(df_review_peek.columns.tolist())

print("\n2️⃣ Kiểu dữ liệu & Missing values:")
missing_info = pd.DataFrame({
    'dtype'        : df_review_peek.dtypes,
    'non_null'     : df_review_peek.count(),
    'missing'      : df_review_peek.isnull().sum(),
    'missing_%'    : (df_review_peek.isnull().sum() / len(df_review_peek) * 100).round(2)
})
print(missing_info)

print("\n3️⃣ Xem 3 dòng mẫu:")
display(df_review_peek.head(3))

## Cell 1.6 — Phân tích cột của Meta dataset

In [ ]:
# ============================================================
# CELL 1.6: Phân tích chi tiết từng cột của Meta
# Meta chứa thông tin sản phẩm: tên, giá, mô tả, category...
# ============================================================

print("=" * 65)
print("📊 META DATASET - PHÂN TÍCH TỪNG CỘT")
print("=" * 65)

print("\n1️⃣ Danh sách cột:")
print(df_meta_peek.columns.tolist())

print("\n2️⃣ Kiểu dữ liệu & Missing values:")
missing_info_meta = pd.DataFrame({
    'dtype'     : df_meta_peek.dtypes,
    'non_null'  : df_meta_peek.count(),
    'missing'   : df_meta_peek.isnull().sum(),
    'missing_%' : (df_meta_peek.isnull().sum() / len(df_meta_peek) * 100).round(2)
})
print(missing_info_meta)

print("\n3️⃣ Xem 3 dòng mẫu:")
display(df_meta_peek.head(3))

## Cell 1.7 — Thống kê mô tả cơ bản

In [ ]:
# ============================================================
# CELL 1.7: Thống kê mô tả cơ bản
# Xem distribution của rating, helpful_vote, độ dài text...
# ============================================================

print("=" * 55)
print("📈 THỐNG KÊ MÔ TẢ - REVIEW DATASET")
print("=" * 55)

# Thống kê Rating
if 'rating' in df_review_peek.columns:
    df_review_peek['rating'] = pd.to_numeric(df_review_peek['rating'], errors='coerce')
    print("\n⭐ Phân bố Rating:")
    print(df_review_peek['rating'].value_counts().sort_index())
    print(f"   Trung bình: {df_review_peek['rating'].mean():.2f}")

# Độ dài text review
if 'text' in df_review_peek.columns:
    df_review_peek['text_length'] = df_review_peek['text'].astype(str).str.len()
    print("\n📝 Độ dài text review (số ký tự):")
    print(df_review_peek['text_length'].describe().round(1))

# Helpful vote
if 'helpful_vote' in df_review_peek.columns:
    df_review_peek['helpful_vote'] = pd.to_numeric(df_review_peek['helpful_vote'], errors='coerce').fillna(0)
    print("\n👍 Helpful votes:")
    print(df_review_peek['helpful_vote'].describe().round(1))

# Verified purchase
if 'verified_purchase' in df_review_peek.columns:
    print("\n✔️  Verified Purchase:")
    print(df_review_peek['verified_purchase'].value_counts())

# Thống kê Meta
print("\n" + "=" * 55)
print("📈 THỐNG KÊ MÔ TẢ - META DATASET")
print("=" * 55)

if 'price' in df_meta_peek.columns:
    df_meta_peek['price_num'] = df_meta_peek['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
    df_meta_peek['price_num'] = pd.to_numeric(df_meta_peek['price_num'], errors='coerce')
    print("\n💲 Phân bố giá sản phẩm ($):")
    print(df_meta_peek['price_num'].describe().round(2))

if 'average_rating' in df_meta_peek.columns:
    print("\n⭐ Average Rating của sản phẩm:")
    print(df_meta_peek['average_rating'].describe().round(2))

## Cell 1.8 — Visualize khám phá ban đầu

In [ ]:
# ============================================================
# CELL 1.8: Visualize EDA sơ bộ
# Vẽ 4 biểu đồ cơ bản để hiểu tổng quan dữ liệu
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bước 1 - Khám phá dữ liệu ban đầu\n(Sample 5,000 dòng đầu)', fontsize=14, fontweight='bold')

# ---- Biểu đồ 1: Phân bố Rating ----
if 'rating' in df_review_peek.columns:
    rating_counts = df_review_peek['rating'].value_counts().sort_index()
    colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
    axes[0,0].bar(rating_counts.index, rating_counts.values, color=colors)
    axes[0,0].set_title('Phân bố Rating (1-5 sao)')
    axes[0,0].set_xlabel('Rating')
    axes[0,0].set_ylabel('Số lượng')
    for i, (idx, val) in enumerate(rating_counts.items()):
        axes[0,0].text(idx, val + 5, str(val), ha='center', fontsize=9)

# ---- Biểu đồ 2: Độ dài text review ----
if 'text_length' in df_review_peek.columns:
    # Giới hạn max 2000 ký tự để dễ nhìn
    text_data = df_review_peek['text_length'].clip(upper=2000)
    axes[0,1].hist(text_data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0,1].set_title('Phân bố độ dài Review (số ký tự)')
    axes[0,1].set_xlabel('Số ký tự (cắt tại 2000)')
    axes[0,1].set_ylabel('Tần suất')
    axes[0,1].axvline(df_review_peek['text_length'].median(), color='red', 
                       linestyle='--', label=f'Median: {df_review_peek["text_length"].median():.0f}')
    axes[0,1].legend()

# ---- Biểu đồ 3: Giá sản phẩm ----
if 'price_num' in df_meta_peek.columns:
    price_data = df_meta_peek['price_num'].dropna()
    price_data = price_data[(price_data > 0) & (price_data < 500)]  # lọc outlier
    axes[1,0].hist(price_data, bins=50, color='#8e44ad', edgecolor='white', alpha=0.8)
    axes[1,0].set_title('Phân bố giá sản phẩm ($)')
    axes[1,0].set_xlabel('Giá ($) - dưới $500')
    axes[1,0].set_ylabel('Tần suất')
    axes[1,0].axvline(price_data.median(), color='red',
                       linestyle='--', label=f'Median: ${price_data.median():.1f}')
    axes[1,0].legend()
else:
    axes[1,0].text(0.5, 0.5, 'Không có dữ liệu giá', ha='center', va='center')
    axes[1,0].set_title('Phân bố giá sản phẩm')

# ---- Biểu đồ 4: Verified Purchase ----
if 'verified_purchase' in df_review_peek.columns:
    vp_counts = df_review_peek['verified_purchase'].value_counts()
    axes[1,1].pie(vp_counts.values, labels=['Verified', 'Not Verified'],
                   colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
                   startangle=90, textprops={'fontsize': 11})
    axes[1,1].set_title('Tỷ lệ Verified Purchase')
else:
    axes[1,1].text(0.5, 0.5, 'Không có dữ liệu\nverified_purchase', ha='center', va='center')

plt.tight_layout()
save_path = FIGURES_DIR / 'step1_initial_exploration.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Đã lưu biểu đồ: {save_path}")

## Cell 1.9 — Tổng kết Bước 1

In [ ]:
# ============================================================
# CELL 1.9: Tổng kết - Ghi lại những gì ta đã học về dữ liệu
# ============================================================

print("=" * 60)
print("✅ TỔNG KẾT BƯỚC 1 - KẾT QUẢ KHÁM PHÁ")
print("=" * 60)

print("\n📋 REVIEW DATASET:")
print(f"  - Các cột có: {df_review_peek.columns.tolist()}")
print(f"  - Cột quan trọng cho dự án:")
print(f"    → 'rating'           : nhãn sentiment (1-5 sao)")
print(f"    → 'text'             : nội dung review (NLP)")
print(f"    → 'user_id'          : định danh người dùng (recommendation)")
print(f"    → 'parent_asin'      : định danh sản phẩm (join với meta)")
print(f"    → 'timestamp'        : thời gian review")
print(f"    → 'helpful_vote'     : review được vote hữu ích")
print(f"    → 'verified_purchase': xác nhận đã mua")

print("\n📋 META DATASET:")
print(f"  - Các cột có: {df_meta_peek.columns.tolist()}")
print(f"  - Cột quan trọng:")
print(f"    → 'parent_asin'  : key để join với review")
print(f"    → 'title'        : tên sản phẩm")
print(f"    → 'price'        : giá sản phẩm")
print(f"    → 'categories'   : danh mục sản phẩm")
print(f"    → 'description'  : mô tả sản phẩm")

print("\n📌 QUAN SÁT QUAN TRỌNG:")
if 'rating' in df_review_peek.columns:
    top_rating = df_review_peek['rating'].value_counts().idxmax()
    print(f"  - Rating phổ biến nhất: {top_rating} sao → dataset có thể imbalanced")
print(f"  - Cần xử lý imbalanced data ở Bước 5 (Sentiment)")
print(f"  - Cần lọc user/item có ít interactions ở Bước 6 (Recommendation)")

print("\n🚀 Sẵn sàng chuyển sang BƯỚC 2: Tiền xử lý!")